# Step 5 - Final Prototype Random Forest

This notebook is the last notebook-stage model before moving the project into a cleaner code structure.

It keeps the model focused:

- Random Forest only
- time-based split
- refined flight features
- top-20 origin airport weather
- small hyperparameter comparison
- threshold comparison


In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

## Load And Clean Flight Data


In [19]:
data_dir = Path("../data/raw/flight/2022")
csv_files = sorted(data_dir.glob("2022-0[1-3]_flight.csv"))

flight_cols = [
    "Month",
    "FlightDate",
    "DayOfWeek",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepDelay",
    "Cancelled"
]

flight_parts = [pd.read_csv(file, usecols=flight_cols) for file in csv_files]
flights = pd.concat(flight_parts, ignore_index=True)
flights.shape

(1598468, 9)

In [20]:
flights = flights[flights["Cancelled"] == 0].copy()

flights = flights.dropna(subset=[
    "FlightDate",
    "CRSDepTime",
    "DepDelay",
    "Reporting_Airline",
    "Origin",
    "Dest"
])

flights = flights.sample(n=200_000, random_state=42)

flights["Delay"] = (flights["DepDelay"] > 15).astype(int)
flights = flights.drop(columns=["DepDelay"])

flights["FlightDate"] = pd.to_datetime(flights["FlightDate"])
flights["CRSDepTime"] = flights["CRSDepTime"].astype(int)
flights["dep_hour"] = flights["CRSDepTime"] // 100

dep_minutes = (flights["CRSDepTime"] // 100) * 60 + (flights["CRSDepTime"] % 100)
flights["scheduled_departure_local"] = flights["FlightDate"] + pd.to_timedelta(dep_minutes, unit="m")

flights["route"] = flights["Origin"] + "_" + flights["Dest"]
flights["is_weekend"] = flights["DayOfWeek"].isin([6, 7]).astype(int)
flights["time_of_day_bin"] = pd.cut(
    flights["dep_hour"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=["overnight", "morning", "afternoon", "evening", "night"]
)

flights.shape

(200000, 14)

## Load Weather And Convert To UTC Join Key


In [21]:
weather_path = Path("../data/raw/weather/asos_top20_origins_2022_01_to_2022_03.csv")
weather = pd.read_csv(weather_path)
weather["valid"] = pd.to_datetime(weather["valid"])
weather.shape

(51923, 15)

In [22]:
weather_stations = sorted(weather["station"].dropna().unique().tolist())
flights = flights[flights["Origin"].isin(weather_stations)].copy()
flights.shape

(104229, 14)

In [23]:
AIRPORT_TIMEZONES = {
    "ATL": "America/New_York",
    "BOS": "America/New_York",
    "CLT": "America/New_York",
    "DCA": "America/New_York",
    "DTW": "America/New_York",
    "EWR": "America/New_York",
    "JFK": "America/New_York",
    "LGA": "America/New_York",
    "MCO": "America/New_York",
    "MIA": "America/New_York",
    "DFW": "America/Chicago",
    "IAH": "America/Chicago",
    "MSP": "America/Chicago",
    "ORD": "America/Chicago",
    "DEN": "America/Denver",
    "PHX": "America/Phoenix",
    "LAS": "America/Los_Angeles",
    "LAX": "America/Los_Angeles",
    "SEA": "America/Los_Angeles",
    "SFO": "America/Los_Angeles"
}

missing_tz = sorted(set(flights["Origin"].unique()) - set(AIRPORT_TIMEZONES.keys()))
missing_tz

[]

In [24]:
def add_scheduled_departure_utc(flight_df, timezone_map):
    pieces = []

    for airport, group in flight_df.groupby("Origin", group_keys=False):
        tz_name = timezone_map.get(airport)
        if tz_name is None:
            raise ValueError(f"Missing timezone for airport: {airport}")

        group = group.copy()
        localized = group["scheduled_departure_local"].dt.tz_localize(
            tz_name,
            nonexistent="shift_forward",
            ambiguous="NaT"
        )
        group["scheduled_departure_utc"] = localized.dt.tz_convert("UTC").dt.tz_localize(None)
        pieces.append(group)

    return pd.concat(pieces, ignore_index=True)


flights = add_scheduled_departure_utc(flights, AIRPORT_TIMEZONES)
flights[["Origin", "scheduled_departure_local", "scheduled_departure_utc"]].head()

,Origin,scheduled_departure_local,scheduled_departure_utc
0,ATL,2022-01-19 08:20:00,2022-01-19 13:20:00
1,ATL,2022-02-02 18:07:00,2022-02-02 23:07:00
2,ATL,2022-01-21 05:30:00,2022-01-21 10:30:00
3,ATL,2022-03-27 16:43:00,2022-03-27 20:43:00
4,ATL,2022-01-11 21:30:00,2022-01-12 02:30:00


## Join Weather


In [25]:
def join_weather_to_flights(flight_df, weather_df, tolerance_hours=2):
    joined_parts = []
    tolerance = pd.Timedelta(hours=tolerance_hours)

    for station in sorted(set(flight_df["Origin"]).intersection(set(weather_df["station"]))):
        flight_part = flight_df[flight_df["Origin"] == station].sort_values("scheduled_departure_utc").copy()
        weather_part = weather_df[weather_df["station"] == station].sort_values("valid").copy()

        merged = pd.merge_asof(
            flight_part,
            weather_part,
            left_on="scheduled_departure_utc",
            right_on="valid",
            direction="backward",
            tolerance=tolerance
        )
        joined_parts.append(merged)

    joined = pd.concat(joined_parts, ignore_index=True)
    joined["weather_report_age_minutes"] = (
        joined["scheduled_departure_utc"] - joined["valid"]
    ).dt.total_seconds().div(60)

    return joined


model_df = join_weather_to_flights(flights, weather, tolerance_hours=2)
model_df.shape

(104229, 31)

In [26]:
model_df["gust"] = model_df["gust"].fillna(0)
model_df["p01i"] = model_df["p01i"].fillna(0)
model_df["vsby"] = model_df["vsby"].fillna(10)
model_df["wxcodes"] = model_df["wxcodes"].fillna("")
model_df["skyc1"] = model_df["skyc1"].fillna("CLR")

model_df["has_precip"] = (model_df["p01i"] > 0).astype(int)
model_df["low_visibility"] = (model_df["vsby"] < 3).astype(int)
model_df["high_wind"] = (model_df["sknt"].fillna(0) >= 15).astype(int)
model_df["has_gust"] = (model_df["gust"] > 0).astype(int)
model_df["has_weather_code"] = (model_df["wxcodes"] != "").astype(int)

required_weather_cols = ["valid", "tmpf", "relh", "sknt", "alti", "weather_report_age_minutes"]
model_df = model_df.dropna(subset=required_weather_cols).copy()

model_df.shape

(103974, 36)

## Time-Based Split


In [27]:
train_df = model_df[model_df["Month"].isin([1, 2])].copy()
test_df = model_df[model_df["Month"] == 3].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train delay rate:", train_df["Delay"].mean())
print("Test delay rate:", test_df["Delay"].mean())

Train shape: (66459, 36)
Test shape: (37515, 36)
Train delay rate: 0.19925066582404188
Test delay rate: 0.21052912168465948


## Add Historical Delay Rate Features


In [28]:
def add_delay_rate_feature(train_data, test_data, group_col, target_col, new_col):
    global_rate = train_data[target_col].mean()
    rates = train_data.groupby(group_col)[target_col].mean()

    train_data[new_col] = train_data[group_col].map(rates).fillna(global_rate)
    test_data[new_col] = test_data[group_col].map(rates).fillna(global_rate)

    return train_data, test_data


train_df, test_df = add_delay_rate_feature(train_df, test_df, "Origin", "Delay", "origin_delay_rate")
train_df, test_df = add_delay_rate_feature(train_df, test_df, "Reporting_Airline", "Delay", "airline_delay_rate")
train_df, test_df = add_delay_rate_feature(train_df, test_df, "route", "Delay", "route_delay_rate")

train_df[["Origin", "Reporting_Airline", "route", "origin_delay_rate", "airline_delay_rate", "route_delay_rate"]].head()

,Origin,Reporting_Airline,route,origin_delay_rate,airline_delay_rate,route_delay_rate
0,ATL,B6,ATL_JFK,0.146075,0.323912,0.219178
1,ATL,AA,ATL_CLT,0.146075,0.178146,0.174757
2,ATL,AA,ATL_DFW,0.146075,0.178146,0.191919
3,ATL,AA,ATL_ORD,0.146075,0.178146,0.153153
4,ATL,DL,ATL_FLL,0.146075,0.143742,0.168750


## Feature Sets

This compares three focused versions of the Random Forest:

- flight-only refined features
- flight plus core weather features
- flight plus weather with extra weather flags


In [29]:
flight_numeric_features = [
    "DayOfWeek",
    "dep_hour",
    "is_weekend",
    "origin_delay_rate",
    "airline_delay_rate",
    "route_delay_rate"
]

flight_categorical_features = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "route",
    "time_of_day_bin"
]

core_weather_numeric_features = [
    "tmpf",
    "relh",
    "sknt",
    "alti",
    "vsby",
    "weather_report_age_minutes",
    "p01i"
]

extra_weather_numeric_features = [
    "gust",
    "has_precip",
    "low_visibility",
    "high_wind",
    "has_gust",
    "has_weather_code"
]

weather_categorical_features = ["skyc1"]

feature_sets = {
    "Flight Only": {
        "numeric": flight_numeric_features,
        "categorical": flight_categorical_features,
    },
    "Flight + Core Weather": {
        "numeric": flight_numeric_features + core_weather_numeric_features,
        "categorical": flight_categorical_features,
    },
    "Flight + Full Weather": {
        "numeric": flight_numeric_features + core_weather_numeric_features + extra_weather_numeric_features,
        "categorical": flight_categorical_features + weather_categorical_features,
    },
}

## Small Random Forest Search

This keeps the search small and compares a few sensible settings.

In [30]:
rf_settings = [
    {"n_estimators": 100, "max_depth": 12, "min_samples_leaf": 5},
    {"n_estimators": 150, "max_depth": 14, "min_samples_leaf": 5},
    {"n_estimators": 150, "max_depth": 16, "min_samples_leaf": 3}
]

In [31]:
def build_random_forest_pipeline(numeric_features, categorical_features, params):
    preprocessor = ColumnTransformer([
        ("num", SimpleImputer(strategy="most_frequent"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    return model


search_results = []
saved_models = {}

for feature_name, spec in feature_sets.items():
    features = spec["numeric"] + spec["categorical"]
    X_train = train_df[features]
    y_train = train_df["Delay"]
    X_test = test_df[features]
    y_test = test_df["Delay"]

    for params in rf_settings:
        model = build_random_forest_pipeline(spec["numeric"], spec["categorical"], params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        result = {
            "feature_set": feature_name,
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "min_samples_leaf": params["min_samples_leaf"],
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_delay": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
            "recall_delay": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
            "f1_delay": f1_score(y_test, y_pred, pos_label=1, zero_division=0)
        }
        search_results.append(result)
        saved_models[(feature_name, params["n_estimators"], params["max_depth"], params["min_samples_leaf"])] = model

search_results_df = pd.DataFrame(search_results).sort_values("f1_delay", ascending=False)
search_results_df

,feature_set,n_estimators,max_depth,min_samples_leaf,accuracy,precision_delay,recall_delay,f1_delay
4,Flight + Core Weather,150,14,5,0.654725,0.314441,0.542289,0.398067
8,Flight + Full Weather,150,16,3,0.657657,0.315444,0.535072,0.396901
5,Flight + Core Weather,150,16,3,0.664108,0.318879,0.524183,0.396533
3,Flight + Core Weather,100,12,5,0.655178,0.313573,0.536465,0.395796
6,Flight + Full Weather,100,12,5,0.641423,0.304904,0.549506,0.392192
7,Flight + Full Weather,150,14,5,0.640490,0.303936,0.548493,0.391134
2,Flight Only,150,16,3,0.636305,0.300236,0.546721,0.387612
1,Flight Only,150,14,5,0.631987,0.298059,0.552038,0.387108
0,Flight Only,100,12,5,0.627776,0.295647,0.555584,0.385928


## Best Model Summary


In [32]:
best_result = search_results_df.iloc[0]
best_result

feature_set         Flight + Core Weather
n_estimators                          150
max_depth                              14
min_samples_leaf                        5
accuracy                         0.654725
precision_delay                  0.314441
recall_delay                     0.542289
f1_delay                         0.398067
Name: 4, dtype: object

In [33]:
best_key = (
    best_result["feature_set"],
    int(best_result["n_estimators"]),
    int(best_result["max_depth"]),
    int(best_result["min_samples_leaf"])
)

best_model = saved_models[best_key]
best_spec = feature_sets[best_result["feature_set"]]
best_features = best_spec["numeric"] + best_spec["categorical"]

X_test_best = test_df[best_features]
y_test_best = test_df["Delay"]
y_pred_best = best_model.predict(X_test_best)

print(classification_report(y_test_best, y_pred_best, zero_division=0))
print(confusion_matrix(y_test_best, y_pred_best))

              precision    recall  f1-score   support

           0       0.85      0.68      0.76     29617
           1       0.31      0.54      0.40      7898

    accuracy                           0.65     37515
   macro avg       0.58      0.61      0.58     37515
weighted avg       0.74      0.65      0.68     37515

[[20279  9338]
 [ 3615  4283]]


## Threshold Comparison

The default threshold is 0.50. This checks a few lower and higher values.

In [34]:
y_prob_best = best_model.predict_proba(X_test_best)[:, 1]

threshold_results = []

for threshold in [0.35, 0.40, 0.45, 0.50, 0.55]:
    y_pred_threshold = (y_prob_best >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_test_best, y_pred_threshold),
        "precision_delay": precision_score(y_test_best, y_pred_threshold, pos_label=1, zero_division=0),
        "recall_delay": recall_score(y_test_best, y_pred_threshold, pos_label=1, zero_division=0),
        "f1_delay": f1_score(y_test_best, y_pred_threshold, pos_label=1, zero_division=0)
    })

pd.DataFrame(threshold_results)

,threshold,accuracy,precision_delay,recall_delay,f1_delay
0,0.35,0.218393,0.211842,0.997088,0.349441
1,0.40,0.301479,0.224698,0.945936,0.363137
2,0.45,0.470505,0.254211,0.783489,0.383871
3,0.50,0.654725,0.314441,0.542289,0.398067
4,0.55,0.763348,0.408752,0.277918,0.330871
